# Pima Indians Diabetes: exploratory data analysis

This notebook uses the real Pima Indians dataset at `data/diabetes.csv`. It explores the source data with Pandas and NumPy, identifies zero values that represent missing measurements, and plots distributions and feature correlations. No fabricated data is generated. If the file is absent, run `python download_data.py` from the project root.

If the CSV is missing, download it from [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) and place it in the project's `data` folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_CANDIDATES = [Path("data") / "diabetes.csv", Path("..") / "data" / "diabetes.csv"]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.is_file()), DATA_CANDIDATES[0].resolve())
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Run `python download_data.py` from the project root."
    )

data = pd.read_csv(DATA_PATH)
data.head()

## Dataset shape, columns, and basic quality

In [ ]:
expected_columns = {
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
}
missing_columns = expected_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")

print(f"Shape: {data.shape}")
data.info()
display(data.describe().round(2))
print("Missing values in source file:")
display(data.isna().sum())
print("Outcome counts:")
display(data["Outcome"].value_counts().sort_index())

## Outcome balance and invalid zero values

Pregnancy count may legitimately be zero. In contrast, zero glucose, blood pressure, skin thickness, insulin, or BMI readings are treated as missing by this project's training pipeline. The cleaning here is for analysis; the reusable correction is included in each training/inference pipeline.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zero values in measurements treated as missing:")
display(data[zero_as_missing].eq(0).sum())

fig, ax = plt.subplots(figsize=(5, 4))
data["Outcome"].value_counts().sort_index().plot.bar(ax=ax, color=["#4C78A8", "#F58518"])
ax.set(title="Outcome class counts", xlabel="Outcome (0 = negative, 1 = positive)", ylabel="Rows")
ax.tick_params(axis="x", rotation=0)
plt.show()

cleaned = data.copy()
cleaned[zero_as_missing] = cleaned[zero_as_missing].replace(0, np.nan)
print("Missing-value counts after invalid-zero correction:")
display(cleaned.isna().sum())

## Feature distributions and correlations

Histograms show the cleaned measurement distributions. Correlations are descriptive associations, not evidence that a feature causes an outcome.

In [ ]:
features = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
cleaned[features].hist(figsize=(14, 10), bins=20, edgecolor="white")
plt.suptitle("Feature distributions after invalid-zero correction")
plt.tight_layout()
plt.show()

correlations = cleaned.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlations.columns)), correlations.columns, rotation=90)
ax.set_yticks(range(len(correlations.columns)), correlations.columns)
ax.set_title("Feature correlation matrix")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()
correlations["Outcome"].sort_values(ascending=False).round(3)

## Next steps

Run `python train.py` from the project root to perform a stratified holdout split, compare Logistic Regression and Random Forest using training-only cross-validation, report the evaluation metrics, and save the winning pipeline. The imputer is fitted inside each pipeline fold to prevent data leakage.